In [7]:
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader

load_dotenv()

file_path = "datasets\한글맞춤법 표준어규정 해설.pdf"

loader = PyPDFLoader(file_path)
pages = []

# async: 비동기 방식으로 실행
# alazy_load(): PDF를 한 번에 읽지 않고 페이지를 하나씩 비동기적으로 가져옴
async for page in loader.alazy_load():
    pages.append(page)

In [8]:
# 각 페이지의 글자 수를 계산해서 리스트로 저장
lengths = [len(page.page_content) for page in pages]

print("페이지 수:", len(lengths))
print("평균:", sum(lengths) / len(lengths))
print("최소:", min(lengths))
print("최대:", max(lengths))

# 페이지 길이가 어느 구간에 많이 분포하는지 확인
for limit in [500, 800, 1000, 1200, 1500, 1800]:
    count = sum(length <= limit for length in lengths)
    print(f"{limit}자 이하: {count}페이지 ({count / len(lengths) * 100:.1f}%)")

페이지 수: 264
평균: 685.5757575757576
최소: 0
최대: 1785
500자 이하: 60페이지 (22.7%)
800자 이하: 180페이지 (68.2%)
1000자 이하: 231페이지 (87.5%)
1200자 이하: 250페이지 (94.7%)
1500자 이하: 260페이지 (98.5%)
1800자 이하: 264페이지 (100.0%)


In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=200)
docs = text_splitter.split_documents(pages)

print("원본 페이지:", len(pages))
print("분할 후 청크:", len(docs))
print("분할 후 청크 글자 수:", [len(i.page_content) for i in docs])


원본 페이지: 264
분할 후 청크: 259
분할 후 청크 글자 수: [63, 793, 528, 548, 343, 1497, 254, 1467, 243, 11, 1492, 466, 640, 976, 957, 1127, 636, 468, 959, 671, 474, 748, 810, 677, 855, 607, 757, 1072, 617, 839, 873, 559, 701, 973, 829, 844, 865, 736, 308, 345, 747, 362, 605, 734, 957, 662, 406, 540, 491, 517, 662, 704, 621, 430, 642, 571, 524, 898, 525, 527, 652, 771, 536, 1030, 514, 1034, 843, 667, 1080, 644, 968, 702, 643, 845, 634, 736, 660, 833, 934, 632, 483, 480, 607, 1022, 590, 939, 615, 419, 423, 778, 512, 580, 573, 666, 569, 864, 544, 602, 553, 407, 381, 807, 987, 501, 720, 641, 563, 572, 525, 622, 603, 894, 853, 517, 777, 563, 752, 506, 779, 417, 379, 535, 515, 564, 429, 443, 488, 524, 553, 559, 502, 728, 535, 541, 591, 503, 611, 499, 575, 466, 551, 485, 533, 12, 1458, 413, 13, 838, 1101, 848, 188, 700, 884, 750, 603, 1070, 773, 809, 632, 716, 1034, 942, 583, 984, 738, 617, 1198, 374, 630, 864, 450, 490, 546, 673, 814, 755, 552, 492, 734, 1323, 715, 694, 367, 551, 733, 437, 634, 682, 652, 788,

In [10]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

DB_PATH = "./chroma_db"

vectorstore = Chroma.from_documents(
    documents= docs,
    embedding= OpenAIEmbeddings(model="text-embedding-3-small"),
    persist_directory=DB_PATH,
    collection_name="korean_pdf"
)

In [11]:
vectorstore.similarity_search("구개음화", k=3)

[Document(id='02e165db-b45c-487f-9004-4f6299700316', metadata={'source': 'datasets\\한글맞춤법 표준어규정 해설.pdf', 'creationdate': '2018-12-20T09:36:29+09:00', 'total_pages': 264, 'title': '<C6ED2DBEEEB9AEB1D4B9FCC7D8BCB35FB1B9BEEEBFF8C3D6C1BE5F76657232302DBCF6C1A4BABB2E687770>', 'producer': 'Acrobat Distiller 9.0.0 (Windows)', 'moddate': '2018-12-20T09:38:09+09:00', 'page_label': '24', 'page': 23, 'creator': 'PScript5.dll Version 5.2.2', 'author': 'admin'}, page_content='‘한글 맞춤법’, ‘표준어 규정’ 해설\x13\x13\n여기서 말하는 ‘종속적(從屬的) 관계’란, 실질 형태소인 체언, 어근, 용언 어간 등에 \n형식 형태소인 조사, 접미사, 어미 등이 결합하는 관계를 말한다. \n솥이[소치]: 솥(실질 형태소)+이(형식 형태소)\n묻히다[무치다]: 묻-(실질 형태소)+-히-(형식 형태소)+-다(형식 형태소)\n구개음화 현상에서 후행하는 형태소는 반드시 조사, 접미사와 같은 형식 형태소이\n어야 한다. 구개음화 현상에 관여하는 형식 형태소에는 다음과 같은 것이 있다.\n이: 명사 파생 접미사(맏이, 해돋이) \n부사 파생 접미사(같이, 굳이)\n주격 조사(끝이, 밭이) \n서술격 조사(끝이다, 밭이다)\n사동 접미사(붙이다)\n히: 피동 접미사(걷히다, 닫히다)\n사동 접미사(굳히다)\n형식 형태소가 결합하지 않은 경우에는 구개음화가 실현되지 않는다. ‘곧이[고지]’\n는 부사 파생 접미사가 결합하여 부사가 되었으므로 구개음화가 실현되었지만, ‘곧이\n어’는 형식 형태소가 아닌 실질 형태소 부사가 결합한 